# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Cleaning dates

In [0]:
df = (
    df
    .withColumn(
        "order_date",
        F.when((col("order_date") == 0) | (length(col("order_date")) != 8), None)
         .otherwise(F.to_date(col("order_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "ship_date",
        F.when((col("ship_date") == 0) | (length(col("ship_date")) != 8), None)
         .otherwise(F.to_date(col("ship_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "due_date",
        F.when((col("due_date") == 0) | (length(col("due_date")) != 8), None)
         .otherwise(F.to_date(col("due_date").cast("string"), "yyyyMMdd"))
    )
)

## Sales and price corrections

In [0]:
df = (
    df
    .withColumn(
        "price",
        F.when(
            (col("price").isNull()) | (col("price") <= 0),
            F.when(
                col("quantity") != 0,
                col("sales_amount") / col("quantity")
            ).otherwise(None)
        ).otherwise(col("price"))
    )
)

## Sanity check of final DataFrame

In [0]:
df.limit(10).display()

# Write into silver table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("workspace.silver.crm_sales")
)